In [2]:
# -*- coding: utf-8 -*-
# CT->MRI(b0) Latent Diffusion + CT latent alignment (L1 + Bucket-SupCon)
# * VAE, (가능한) U-Net은 CompVis LDM ckpt에서 weight 로드
# * 새로 학습: E_ct(CT->latent), Projectors, (U-Net은 선택적으로 joint finetune)

import os, glob, time, random, warnings
from typing import List, Tuple, Dict, Optional
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import sys
sys.path.append('.')  # ensure local ldm/ importable

from ldm.modules.diffusionmodules.openaimodel import UNetModel
try:
    from ldm.modules.diffusionmodules.util import (
        make_beta_schedule, make_ddim_timesteps, make_ddim_sampling_parameters
    )
except Exception:
    from ldm.modules.diffusionmodules.utils import (
        make_beta_schedule, make_ddim_timesteps, make_ddim_sampling_parameters
    )
from ldm.util import instantiate_from_config

In [3]:
# ===================== Config =====================
CFG = {
    "seed": 42,
    "device": "cuda" if torch.cuda.is_available() else "cpu",

    # 데이터
    "data_root": "/data/ct-mri/dataset/processed_final_5",
    "use_groups": ["bleed", "non_bleed"],
    "ct_dir_candidates": ["ct_axial","ct","ct_png","ctjpg","ctjpeg"],
    "mri_b0_dir_candidates": ["mri_b0_axial","mri_b0","b0","dwi_b0","dwi","dwi_axial_b0"],
    "ct_file_globs": ["*ct*.jpg","*CT*.jpg"],
    "mri_b0_file_globs": ["*b0*.jpg","*B0*.jpg","*dwi*.jpg","*DWI*.jpg"],

    # 학습
    "img_size": 256,
    "batch_size": 8,
    "num_workers": 4,
    "epochs": 50,
    "lr": 2e-4,
    "ema_decay": 0.999,

    # 확산 스케줄
    "num_timesteps": 1000,
    "beta_schedule": "cosine",
    "ddim_steps": 200,
    "ddim_eta": 0.0,

    # ----- LDM 가중치 경로 -----
    "ldm_ckpt": "ldm_model.ckpt",  # ← CompVis LDM ckpt (diffusion_model.* 포함)
    "vae": {
        "config": "autoencoder_kl_64x64x3.yaml",   # ← KL/VQ yaml
        "ckpt":   "ae_model.ckpt", # ← first-stage 포함 ckpt (동일 파일 OK)
        "eval_mode": True,
        "freeze": True
    },

    # ----- 잠재 U-Net & cond -----
    "latent_unet": {
        "z_channels": 3,         # yaml과 자동 동기화됨
        "cond_channels": 3,      # E_ct 출력도 zc와 같게(강한 concat)
        "model_channels": 192,
        "num_res_blocks": 2,
        "channel_mult": (1,2,3,4),
        "attention_resolutions": (16,),
        "num_heads": -1,
        "num_head_channels": 64,
        "num_heads_upsample": -1,
        "dropout": 0.1,
        "use_checkpoint": False,
        "use_fp16": False,
        "use_scale_shift_norm": True,
        "resblock_updown": False,
        "use_new_attention_order": False,
        "use_spatial_transformer": False,
        "transformer_depth": 1,
        "context_dim": None,
        "n_embed": None,
        "legacy": True
    },

    # ----- 정렬/대조 하이퍼 -----
    "align_weight": 0.03,  # λ_align
    "contrast": {
        "use": True,
        "weight": 0.07,           # λ_con
        "temperature": 0.07,
        "embed_dim": 128,
        "pool": "global",
        "mode": "B",              # "A": same patient & bucket, "B": bucket only
        "n_buckets": 3,
        "boundaries": [0.20, 0.80],  # low|mid, mid|top
        "soft_band": 0.02
    },

    # ----- 학습 스케줄 옵션 -----
    "warmup_freeze_unet_epochs": 0,  # >0이면 초기 epochs 동안 U-Net freeze (E_ct/Proj만 학습)

    # 저장/인퍼런스
    "save_dir": "./ckpt_ct2mri_b0_ldm_align_bucket_preinit",
    "resume": None,
    "inference": {
        "enabled": True,
        "every_n_epochs": 10,
        "num_examples": 8,
        "sampler": "ddim",
        "ddim_steps": 200,
        "eta": 0.0,
        "seed": 1234,
        "ct_list": [],
        "ct_glob": "/data/ct-mri/dataset/processed_final_5/eval/*/ct_axial/*.jpg",
        "out_subdir": "epoch_infer_align",
        "write_mapping": True,
        "use_ct_init": True    # 정렬 안정화 후 True 추천
    }
}

In [4]:
# ===================== Utils =====================
def set_seed(s:int):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def imread_gray(path: str, size: int) -> np.ndarray:
    img = Image.open(path).convert("L").resize((size,size), Image.BICUBIC)
    return np.asarray(img, dtype=np.float32)/255.0

def to_tensor01(arr: np.ndarray) -> torch.Tensor:
    return torch.from_numpy(arr)[None,...]

def to_range_m11(x01: torch.Tensor) -> torch.Tensor:
    return x01*2.0 - 1.0

def to_image_from_m11(x: torch.Tensor) -> Image.Image:
    x = x.detach().clamp(-1,1); x = (x+1.0)*0.5
    return Image.fromarray((x[0].cpu().numpy()*255).round().astype(np.uint8), "L")

def save_triptych(ct_m11, pred_m11, gt_path, out_path: str, size: int):
    ct_p = to_image_from_m11(ct_m11)
    pred_p = to_image_from_m11(pred_m11)
    if gt_path and os.path.exists(gt_path):
        gt = Image.fromarray((imread_gray(gt_path, size)*255).astype(np.uint8), "L")
    else:
        gt = Image.new("L", (size,size), 0)
    panel = Image.new("L", (size*3, size))
    panel.paste(ct_p,(0,0)); panel.paste(gt,(size,0)); panel.paste(pred_p,(size*2,0))
    os.makedirs(os.path.dirname(out_path), exist_ok=True); panel.save(out_path)

def _find_first_existing_dir(base: str, candidates_lower: List[str]) -> str:
    entries = [d for d in glob.glob(os.path.join(base,"*")) if os.path.isdir(d)]
    name2path = {os.path.basename(p).lower(): p for p in entries}
    for cand in candidates_lower:
        if cand in name2path: return name2path[cand]
    return ""

def _glob_files(base: str, patterns: List[str]) -> List[str]:
    out=[]
    for pat in patterns: out.extend(glob.glob(os.path.join(base, pat)))
    return sorted(set(out))

def _collect_modality_files(patient_dir: str, dir_candidates: List[str], file_globs: List[str]) -> List[str]:
    found = _find_first_existing_dir(patient_dir, [c.lower() for c in dir_candidates])
    if found:
        files = sorted(glob.glob(os.path.join(found,"*.jpg")))
        if files: return files
    return _glob_files(patient_dir, file_globs)

def _pair_by_filename(ct_files: List[str], mri_files: List[str]) -> List[Tuple[str,str]]:
    m = {os.path.splitext(os.path.basename(p))[0]: p for p in mri_files}
    pairs=[]
    for c in ct_files:
        stem = os.path.splitext(os.path.basename(c))[0]
        if stem in m: pairs.append((c, m[stem]))
    return pairs

def discover_pairs(data_root: str, use_groups: List[str], ct_dir_candidates, mri_b0_dir_candidates, ct_file_globs, mri_b0_file_globs):
    all_pairs=[]
    for g in use_groups:
        gdir=os.path.join(data_root,g)
        if not os.path.isdir(gdir): continue
        patients=sorted([p for p in glob.glob(os.path.join(gdir,"*")) if os.path.isdir(p)])
        for pd in patients:
            ct_files  = _collect_modality_files(pd, ct_dir_candidates,  ct_file_globs)
            mri_files = _collect_modality_files(pd, mri_b0_dir_candidates, mri_b0_file_globs)
            if not ct_files or not mri_files: continue
            pairs=_pair_by_filename(ct_files,mri_files)
            if not pairs:
                warnings.warn(f"[Skip patient - no filename match] {pd}")
                continue
            all_pairs.extend(pairs)
    if not all_pairs: raise RuntimeError(f"No CT-MRI(b0) pairs under {data_root}")
    return all_pairs

class CTMRIFolderDataset(Dataset):
    def __init__(self, cfg: Dict):
        self.cfg=cfg
        self.pairs=discover_pairs(cfg["data_root"], cfg["use_groups"], cfg["ct_dir_candidates"], cfg["mri_b0_dir_candidates"], cfg["ct_file_globs"], cfg["mri_b0_file_globs"])
        self.size=cfg["img_size"]
        print(f"[Dataset] discovered pairs: {len(self.pairs)}")

        # 환자별 슬라이스 목록/정규화
        self.pid2files={}
        for ct_path,_ in self.pairs:
            pid=os.path.basename(os.path.dirname(os.path.dirname(ct_path)))
            self.pid2files.setdefault(pid,[]).append(ct_path)
        for pid in self.pid2files:
            self.pid2files[pid]=sorted(self.pid2files[pid])

        self.pid2minmax={}
        for pid, files in self.pid2files.items():
            idxs=[]
            for p in files:
                stem=os.path.splitext(os.path.basename(p))[0]
                try: idxs.append(int(stem))
                except: idxs.append(None)
            if any(v is None for v in idxs):
                idxs=list(range(len(files)))
            v=np.array(idxs,dtype=np.float32)
            self.pid2minmax[pid]=(float(v.min()), float(v.max()))

    def __len__(self): return len(self.pairs)

    def __getitem__(self, i):
        ct_path, mri_path = self.pairs[i]
        pid  = os.path.basename(os.path.dirname(os.path.dirname(ct_path)))
        stem = os.path.splitext(os.path.basename(ct_path))[0]
        files = self.pid2files[pid]
        try: sidx=int(stem)
        except: sidx=files.index(ct_path)

        mn,mx=self.pid2minmax[pid]
        idx_norm=0.0 if mx<=mn else (float(sidx)-mn)/(mx-mn+1e-6)

        ct  = to_range_m11(to_tensor01(imread_gray(ct_path,  self.size)))
        mri = to_range_m11(to_tensor01(imread_gray(mri_path, self.size)))
        return {"ct_m11":ct, "mri_m11":mri, "ct_path":ct_path, "mri_path":mri_path, "pid":pid, "slice_idx":sidx, "idx_norm":idx_norm}


In [5]:
# ===================== VAE (KL/VQ) =====================
class VAEWrapper(nn.Module):
    def __init__(self, enc_fn, dec_fn): super().__init__(); self._enc=enc_fn; self._dec=dec_fn
    @torch.no_grad() 
    def encode(self, x): 
        return self._enc(x)
    @torch.no_grad() 
    def decode(self, z): 
        return self._dec(z)

def build_vae(cfg):
    vae_cfg=cfg["vae"]; device=cfg["device"]
    from omegaconf import OmegaConf
    raw=OmegaConf.load(vae_cfg["config"]); conf=OmegaConf.to_container(raw, resolve=True)
    ckpt_path=vae_cfg["ckpt"]; assert os.path.exists(ckpt_path), f"VAE ckpt not found: {ckpt_path}"
    sd_all=torch.load(ckpt_path, map_location="cpu"); state=sd_all.get("state_dict", sd_all)
    def strip_fs(st):
        if any(k.startswith("first_stage_model.") for k in st.keys()):
            return {k[len("first_stage_model."):]: v for k,v in st.items() if k.startswith("first_stage_model.")}
        return st
    top=conf.get("model",{}); target=top.get("target","")

    if "AutoencoderKL" in target:
        try:
            from ldm.models.autoencoder import AutoencoderKL
        except Exception:
            from ldm.models.autoencoder_kl import AutoencoderKL
        kl=instantiate_from_config(top).to(device)
        state=strip_fs(state); kl.load_state_dict(state, strict=False)
        if vae_cfg.get("eval_mode",True): kl.eval()
        if vae_cfg.get("freeze",True):
            for p in kl.parameters(): p.requires_grad_(False)
        scale=0.18215
        def _enc(x):
            x=x.repeat(1,3,1,1) if x.size(1)==1 else x
            posterior=kl.encode(x); return posterior.mean*scale    # mean 사용(안정)
        def _dec(z):
            x=kl.decode(z/scale)
            # 밝기 가중 Y로 변환해 1ch
            x01=(x+1.0)*0.5
            y01=0.299*x01[:,0:1]+0.587*x01[:,1:2]+0.114*x01[:,2:3]
            return y01*2.0-1.0
        zc=top["params"]["ddconfig"]["z_channels"]
        if cfg["latent_unet"]["z_channels"]!=zc:
            print(f"[VAE/KL] z_channels {cfg['latent_unet']['z_channels']} -> {zc}")
            cfg["latent_unet"]["z_channels"]=zc
        return VAEWrapper(_enc,_dec)

    # else: first_stage_config (KL/VQ)도 지원(생략) — 필요시 위 KL과 동일하게 작성
    raise RuntimeError("VAE yaml 상단에 AutoencoderKL이 필요합니다.")


In [6]:
# ===================== Diffusion schedule =====================
class Sched:
    def __init__(self, betas, alphas, a_bar, a_bar_prev):
        self.betas=betas; self.alphas=alphas
        self.a_bar=a_bar; self.a_bar_prev=a_bar_prev
        self.sqrt_a_bar=torch.sqrt(a_bar)
        self.sqrt_one_minus_a_bar=torch.sqrt(1.0 - a_bar)
        self.post_var=betas*(1.0-a_bar_prev)/(1.0-a_bar)

def build_schedule(num_steps: int, beta_schedule="cosine", device="cpu"):
    betas=torch.from_numpy(make_beta_schedule(beta_schedule, num_steps)).float().to(device)
    alphas=1.0 - betas
    a_bar=torch.cumprod(alphas, dim=0)
    a_bar_prev=torch.cat([torch.tensor([1.0], device=device), a_bar[:-1]], dim=0)
    return Sched(betas, alphas, a_bar, a_bar_prev)

def q_sample_latent(z0, t, sched: Sched, noise=None):
    if noise is None: noise=torch.randn_like(z0)
    sa=sched.sqrt_a_bar[t].view(-1,1,1,1); so=sched.sqrt_one_minus_a_bar[t].view(-1,1,1,1)
    return sa*z0 + so*noise, noise


In [7]:
# ===================== Models =====================
def build_latent_unet(cfg, lat_hw: int):
    lu=cfg["latent_unet"]
    in_ch=lu["z_channels"] + lu["cond_channels"]
    out_ch=lu["z_channels"]
    net=UNetModel(
        image_size=lat_hw, in_channels=in_ch, out_channels=out_ch,
        model_channels=lu["model_channels"], num_res_blocks=lu["num_res_blocks"],
        channel_mult=tuple(lu["channel_mult"]),
        attention_resolutions=tuple(lu["attention_resolutions"]),
        num_heads=lu.get("num_heads",-1), num_head_channels=lu.get("num_head_channels",64),
        num_heads_upsample=lu.get("num_heads_upsample",-1), dropout=lu["dropout"],
        conv_resample=True, dims=2, num_classes=None,
        use_checkpoint=lu["use_checkpoint"], use_fp16=lu["use_fp16"],
        use_scale_shift_norm=lu["use_scale_shift_norm"], resblock_updown=lu["resblock_updown"],
        use_new_attention_order=lu["use_new_attention_order"],
        use_spatial_transformer=lu["use_spatial_transformer"],
        transformer_depth=lu["transformer_depth"], context_dim=lu["context_dim"], n_embed=lu["n_embed"], legacy=lu["legacy"]
    )
    return net

def load_unet_from_ldm_ckpt(net: nn.Module, ldm_ckpt_path: str):
    """ CompVis LDM ckpt에서 diffusion_model.*을 골라 **형상 일치 항목만** 로드 """
    assert os.path.exists(ldm_ckpt_path), f"ldm_ckpt not found: {ldm_ckpt_path}"
    sd_all=torch.load(ldm_ckpt_path, map_location="cpu"); state=sd_all.get("state_dict", sd_all)
    # diffusion_model. prefix
    raw={k[len("diffusion_model."):]: v for k,v in state.items() if k.startswith("diffusion_model.")}
    cur=net.state_dict()
    filt={k:v for k,v in raw.items() if (k in cur and cur[k].shape==v.shape)}
    missing=set(cur.keys())-set(filt.keys())
    unexpected=set(raw.keys())-set(filt.keys())
    net.load_state_dict(filt, strict=False)
    print(f"[U-Net preload] loaded {len(filt)} tensors from LDM; skipped(mismatch) {len(missing)}; extra in ckpt {len(unexpected)}")
    return missing, unexpected

class CTLatentEncoder(nn.Module):
    def __init__(self, z_channels=3):
        super().__init__()
        ch=64
        self.net=nn.Sequential(
            nn.Conv2d(1, ch, 3, 2, 1), nn.SiLU(),     # 256->128
            nn.Conv2d(ch, ch*2, 3, 2, 1), nn.SiLU(),  # 128->64
            nn.Conv2d(ch*2, z_channels, 3, 1, 1)     # (B,zc,64,64)
        )
    def forward(self, x_m11): return self.net(x_m11)

class LatentProjector(nn.Module):
    def __init__(self, in_ch: int, embed_dim: int = 128, pool: str = "global"):
        super().__init__()
        self.pool=pool
        if pool=="global":
            self.head=nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(in_ch, embed_dim, 1))
        elif pool=="patch":
            self.head=nn.Conv2d(in_ch, embed_dim, 1)
        else:
            raise ValueError("pool must be 'global' or 'patch'")
    def forward(self, z):
        if self.pool=="global":
            e=self.head(z).squeeze(-1).squeeze(-1); e=nn.functional.normalize(e, dim=1); return e,None
        else:
            e=self.head(z); B,D,H,W=e.shape
            e=e.permute(0,2,3,1).reshape(B,H*W,D); e=nn.functional.normalize(e, dim=-1); return e,(H,W)

def make_bucket_soft_labels(idx_norm: torch.Tensor, boundaries=(0.20,0.80), soft_band=0.02):
    lo,hi=boundaries; B=idx_norm.numel()
    hard=torch.zeros(B, dtype=torch.long, device=idx_norm.device)
    hard[idx_norm>=lo]=1; hard[idx_norm>=hi]=2
    soft=torch.zeros(B,3, device=idx_norm.device); soft[torch.arange(B),hard]=1.0
    loL,loR=lo-soft_band, lo+soft_band; hiL,hiR=hi-soft_band, hi+soft_band
    mask_lo=(idx_norm>=loL)&(idx_norm<=loR); mask_hi=(idx_norm>=hiL)&(idx_norm<=hiR)
    if mask_lo.any():
        t=(idx_norm[mask_lo]-loL)/(loR-loL+1e-6); soft[mask_lo,0]=1.0-t; soft[mask_lo,1]=t; soft[mask_lo,2]=0.0
    if mask_hi.any():
        t=(idx_norm[mask_hi]-hiL)/(hiR-hiL+1e-6); soft[mask_hi,1]=1.0-t; soft[mask_hi,2]=t; soft[mask_hi,0]=0.0
    soft=soft/(soft.sum(dim=1,keepdim=True)+1e-8)
    return hard, soft

def supervised_contrastive_soft_loss(e_ct, e_mri, soft_bucket: torch.Tensor, pid_list, mode="B", temperature=0.07):
    e_ct=nn.functional.normalize(e_ct, dim=1); e_mri=nn.functional.normalize(e_mri, dim=1)
    B,_=e_ct.shape; E=torch.cat([e_ct,e_mri],0); logits=(E@E.t())/temperature
    eye=torch.eye(2*B, device=E.device).bool(); logits=logits.masked_fill(eye,-1e9)
    soft=torch.cat([soft_bucket, soft_bucket],0)
    if mode.upper()=="A":
        uniq={p:i for i,p in enumerate(sorted(set(pid_list)))}; pid_vec=torch.tensor([uniq[p] for p in pid_list], device=E.device)
        pid2=torch.cat([pid_vec,pid_vec],0); same_pid=(pid2.unsqueeze(0)==pid2.unsqueeze(1))
    else:
        same_pid=torch.ones((2*B,2*B), dtype=torch.bool, device=E.device)
    bucket_sim=(soft@soft.t()); pos_w=bucket_sim*same_pid; pos_w=pos_w.masked_fill(eye,0.0)
    log_prob=logits - torch.logsumexp(logits, dim=1, keepdim=True)
    pos_sum=pos_w.sum(dim=1)+1e-8
    loss_vec=-(log_prob*pos_w).sum(dim=1)/pos_sum
    loss_vec=torch.where(pos_sum>1e-7, loss_vec, torch.zeros_like(loss_vec))
    return loss_vec.mean()

In [8]:
# ===================== EMA =====================
class EMA:
    def __init__(self, model, decay=0.999):
        self.decay=decay; self.shadow={}; self.backup={}
        for n,p in model.named_parameters():
            if p.requires_grad: self.shadow[n]=p.data.clone()
    def update(self, model):
        for n,p in model.named_parameters():
            if p.requires_grad:
                self.shadow[n]=(1.0-self.decay)*p.data + self.decay*self.shadow[n]
    def apply_shadow(self, model):
        self.backup={}
        for n,p in model.named_parameters():
            if p.requires_grad:
                self.backup[n]=p.data.clone(); p.data=self.shadow[n]
    def restore(self, model):
        for n,p in model.named_parameters():
            if p.requires_grad and n in self.backup:
                p.data=self.backup[n]
        self.backup={}


In [9]:
# ===================== Inference utils =====================
@torch.no_grad()
def sample_ddim_latent_aligned(net, E_ct, vae, ct_m11, sched: Sched, ddim_steps=200, eta=0.0, device="cuda", use_ct_init=True):
    B=ct_m11.size(0); z_ct=E_ct(ct_m11); Cz,h,w=z_ct.shape[1],z_ct.shape[2],z_ct.shape[3]
    z = z_ct + torch.randn(B,Cz,h,w,device=device) if use_ct_init else torch.randn(B,Cz,h,w,device=device)
    ts=np.array(make_ddim_timesteps('uniform', ddim_steps, sched.betas.size(0))) - 1
    sigmas, alphas, alphas_prev = make_ddim_sampling_parameters(sched.a_bar.cpu().numpy(), ts, eta)
    sigmas=torch.from_numpy(sigmas).float().to(device); alphas=torch.from_numpy(alphas).float().to(device); alphas_prev=torch.from_numpy(alphas_prev).float().to(device)
    for i in reversed(range(len(ts))):
        t=int(ts[i]); t_b=torch.full((B,), t, device=device, dtype=torch.long)
        eps=net(torch.cat([z, z_ct],1).float(), t_b)
        a_t,a_prev,s_t=alphas[i],alphas_prev[i],sigmas[i]
        z0=(z - torch.sqrt(1.0 - a_t)*eps)/torch.sqrt(a_t)
        dir_term=torch.sqrt(1.0 - a_prev - s_t**2)*eps
        noise = s_t*torch.randn_like(z) if i>0 else 0.0
        z = torch.sqrt(a_prev)*z0 + dir_term + noise
    return vae.decode(z)

def find_eval_gt(p: str) -> Optional[str]:
    """ eval/<PID>/ct_axial/<stem>.jpg -> sibling mri_b0_axial/<stem>.jpg """
    ct_dir=os.path.dirname(p); pid_dir=os.path.dirname(ct_dir); stem=os.path.splitext(os.path.basename(p))[0]
    cand=os.path.join(pid_dir,"mri_b0_axial",f"{stem}.jpg")
    return cand if os.path.exists(cand) else None

@torch.no_grad()
def run_epoch_inference_align(net, E_ct, vae, sched: Sched, cfg, ds, device, epoch_idx):
    inf = cfg["inference"]
    if not inf.get("enabled", False):
        return
    if inf["seed"] is not None:
        torch.manual_seed(inf["seed"]); torch.cuda.manual_seed_all(inf["seed"])

    # CT 목록 수집
    if inf["ct_list"]:
        paths = inf["ct_list"]
    elif inf["ct_glob"]:
        paths = sorted(glob.glob(inf["ct_glob"]))
    else:
        n = min(inf["num_examples"], len(ds))
        paths = [ds.pairs[i][0] for i in range(n)]

    H = cfg["img_size"]
    outdir = os.path.join(cfg["save_dir"], inf["out_subdir"])
    os.makedirs(outdir, exist_ok=True)

    B = 4
    for i in range(0, len(paths), B):
        batch = paths[i:i+B]
        ct_m11 = torch.stack(
            [to_range_m11(to_tensor01(imread_gray(p, H))) for p in batch], dim=0
        ).to(device)

        # 생성
        if inf["sampler"].lower() == "ddpm":
            pred = sample_ddpm_latent_aligned(
                net, E_ct, vae, ct_m11, sched, device=device, use_ct_init=inf["use_ct_init"]
            )
            tag = "ddpm"
        else:
            pred = sample_ddim_latent_aligned(
                net, E_ct, vae, ct_m11, sched,
                ddim_steps=inf["ddim_steps"], eta=inf["eta"],
                device=device, use_ct_init=inf["use_ct_init"]
            )
            tag = f"ddim_eta{inf['eta']:.1f}"

        # 저장 + GT 붙이기 (여기서 p 사용)
        for k, p in enumerate(batch):
            stem = os.path.splitext(os.path.basename(p))[0]
            pid  = os.path.basename(os.path.dirname(os.path.dirname(p)))
            base = f"ep{epoch_idx:03d}_{pid}_{stem}_{tag}"

            # 예측 저장
            to_image_from_m11(pred[k]).save(os.path.join(outdir, f"{base}.png"))

            # GT 경로: 형제 폴더 탐색 (eval/<PID>/ct_axial/<stem>.jpg → mri_b0_axial/<stem>.jpg)
            ct_dir  = os.path.dirname(p)                      # .../ct_axial
            pid_dir = os.path.dirname(ct_dir)                 # .../<PID>
            gt_cand = os.path.join(pid_dir, "mri_b0_axial", f"{stem}.jpg")
            gt_path = gt_cand if os.path.exists(gt_cand) else None

            # triptych 저장
            trip = os.path.join(outdir, f"{base}_triptych.png")
            save_triptych(ct_m11[k], pred[k], gt_path, trip, size=H)

    # (옵션) 매핑 파일
    if inf.get("write_mapping", False):
        with open(os.path.join(outdir, f"ep{epoch_idx:03d}_mapping.txt"), "w") as f:
            for p in paths:
                stem = os.path.splitext(os.path.basename(p))[0]
                pid  = os.path.basename(os.path.dirname(os.path.dirname(p)))
                f.write(f"{pid}_{stem} <- {p}\n")


In [10]:
@torch.no_grad()
def run_epoch_inference_align(net, E_ct, vae, sched: Sched, cfg, ds, device, epoch_idx):
    inf = cfg["inference"]
    if not inf.get("enabled", False):
        return

    # 고정 시드(옵션)
    if inf.get("seed") is not None:
        torch.manual_seed(inf["seed"]); torch.cuda.manual_seed_all(inf["seed"])

    # 1) CT 목록 수집
    if inf.get("ct_list"):
        paths = inf["ct_list"]
    elif inf.get("ct_glob"):
        paths = sorted(glob.glob(inf["ct_glob"]))
    else:
        n = min(inf.get("num_examples", 8), len(ds))
        paths = [ds.pairs[i][0] for i in range(n)]

    if len(paths) == 0:
        print("[Infer] No CT paths found. Check ct_glob / ct_list.")
        return

    H = cfg["img_size"]
    outdir = os.path.join(cfg["save_dir"], inf.get("out_subdir", "epoch_infer"))
    os.makedirs(outdir, exist_ok=True)

    # 2) 배치 단위로 생성
    B = 4
    use_ct_init = bool(inf.get("use_ct_init", True))
    sampler = (inf.get("sampler", "ddim") or "ddim").lower()
    ddim_steps = int(inf.get("ddim_steps", 200))
    eta = float(inf.get("eta", 0.0))

    for i in range(0, len(paths), B):
        batch = paths[i:i+B]
        ct_m11 = torch.stack([to_range_m11(to_tensor01(imread_gray(p, H))) for p in batch], dim=0).to(device)

        # 생성
        if sampler == "ddpm":
            pred = sample_ddpm_latent_aligned(net, E_ct, vae, ct_m11, sched, device=device, use_ct_init=use_ct_init)
            tag = "ddpm"
        else:
            pred = sample_ddim_latent_aligned(net, E_ct, vae, ct_m11, sched,
                                              ddim_steps=ddim_steps, eta=eta,
                                              device=device, use_ct_init=use_ct_init)
            tag = f"ddim_eta{eta:.1f}"

        # 3) 저장 + GT(형제 폴더 탐색) 붙이기
        for k, p in enumerate(batch):
            stem = os.path.splitext(os.path.basename(p))[0]
            pid  = os.path.basename(os.path.dirname(os.path.dirname(p)))
            base = f"ep{epoch_idx:03d}_{pid}_{stem}_{tag}"

            # pred 저장
            to_image_from_m11(pred[k]).save(os.path.join(outdir, f"{base}.png"))

            # GT 경로: eval/<PID>/ct_axial/<stem>.jpg → eval/<PID>/mri_b0_axial/<stem>.jpg
            ct_dir  = os.path.dirname(p)                       # .../ct_axial
            pid_dir = os.path.dirname(ct_dir)                  # .../<PID>
            gt_cand = os.path.join(pid_dir, "mri_b0_axial", f"{stem}.jpg")
            gt_path = gt_cand if os.path.exists(gt_cand) else None

            # triptych 저장
            save_triptych(ct_m11[k], pred[k], gt_path,
                          os.path.join(outdir, f"{base}_triptych.png"),
                          size=H)

    # 4) 매핑 파일(옵션)
    if inf.get("write_mapping", False):
        with open(os.path.join(outdir, f"ep{epoch_idx:03d}_mapping.txt"), "w") as f:
            for p in paths:
                stem = os.path.splitext(os.path.basename(p))[0]
                pid  = os.path.basename(os.path.dirname(os.path.dirname(p)))
                f.write(f"{pid}_{stem} <- {p}\n")


In [11]:
# ===================== Main =====================
def main(cfg=CFG):
    set_seed(cfg["seed"]); device=cfg["device"]; os.makedirs(cfg["save_dir"], exist_ok=True)
    if not cfg.get("resume"):
        last=os.path.join(cfg["save_dir"],"last.pt"); cfg["resume"]=last if os.path.exists(last) else None

    ds=CTMRIFolderDataset(cfg)
    dl=DataLoader(ds, batch_size=cfg["batch_size"], shuffle=True, num_workers=cfg["num_workers"], pin_memory=True, drop_last=True)

    # Teacher VAE
    vae=build_vae(cfg)

    # latent size
    with torch.no_grad():
        tmp=next(iter(dl)); z0=vae.encode(tmp["mri_m11"].to(device)); lat_hw=z0.shape[-2]

    # U-Net(잠재) 만들고 → CompVis LDM ckpt로 부분 로드(형상 맞는 것만)
    net=build_latent_unet(cfg, lat_hw).to(device)
    missing,_=load_unet_from_ldm_ckpt(net, cfg["ldm_ckpt"])

    # CT latent encoder + Projectors
    zc=cfg["latent_unet"]["z_channels"]
    E_ct=CTLatentEncoder(z_channels=zc).to(device)
    proj_dim=cfg["contrast"]["embed_dim"]; pool=cfg["contrast"]["pool"]
    Proj_ct = LatentProjector(in_ch=zc, embed_dim=proj_dim, pool=pool).to(device)
    Proj_mri= LatentProjector(in_ch=zc, embed_dim=proj_dim, pool=pool).to(device)

    # schedule/optim/ema
    sched=build_schedule(cfg["num_timesteps"], cfg["beta_schedule"], device)
    opt=optim.AdamW(list(net.parameters()) + list(E_ct.parameters()) + list(Proj_ct.parameters()) + list(Proj_mri.parameters()), lr=cfg["lr"])
    ema=EMA(net, decay=cfg["ema_decay"])

    start_epoch=0
    if cfg["resume"] and os.path.exists(cfg["resume"]):
        ck=torch.load(cfg["resume"], map_location=device)
        net.load_state_dict(ck["model"], strict=False)
        E_ct.load_state_dict(ck["ct_encoder"], strict=False)
        Proj_ct.load_state_dict(ck["proj_ct"], strict=False)
        Proj_mri.load_state_dict(ck["proj_mri"], strict=False)
        if ck.get("ema", None) is not None:
            for n,p in net.named_parameters():
                if p.requires_grad and n in ck["ema"]:
                    ema.shadow[n]=ck["ema"][n].clone().to(p.device)
        if "opt" in ck: opt.load_state_dict(ck["opt"])
        start_epoch=ck.get("epoch",0)
        print(f"[Resume] {cfg['resume']} @ epoch {start_epoch}")

    mse=nn.MSELoss()

    net.train(); E_ct.train(); Proj_ct.train(); Proj_mri.train()
    for epoch in range(start_epoch, cfg["epochs"]):
        t0=time.time()

        # Warmup: U-Net freeze (선택)
        freeze_unet = (epoch < cfg["warmup_freeze_unet_epochs"])
        for p in net.parameters(): p.requires_grad_(not freeze_unet)

        for step,batch in enumerate(dl):
            ct=batch["ct_m11"].to(device)
            mri=batch["mri_m11"].to(device)

            # teacher latent (mean 사용: 안정)
            with torch.no_grad():
                z_mri=vae.encode(mri)    # (B,zc,h,w)

            # student latent
            z_ct=E_ct(ct)               # (B,zc,h,w)

            # L1 alignment
            l_align=(z_ct - z_mri).abs().mean()

            # Bucket SupCon
            if cfg["contrast"]["use"]:
                idx_norm=torch.as_tensor(batch["idx_norm"], dtype=torch.float32, device=device)
                if idx_norm.dim()==0: idx_norm=idx_norm.repeat(ct.size(0))
                _, soft=make_bucket_soft_labels(idx_norm, boundaries=tuple(cfg["contrast"]["boundaries"]), soft_band=cfg["contrast"]["soft_band"])
                e_ct,_  = Proj_ct(z_ct)
                e_mri,_ = Proj_mri(z_mri)
                pid_list = batch["pid"] if isinstance(batch["pid"], list) else list(batch["pid"])
                l_con = supervised_contrastive_soft_loss(e_ct, e_mri, soft, pid_list, mode=cfg["contrast"]["mode"], temperature=cfg["contrast"]["temperature"])
            else:
                l_con = torch.tensor(0.0, device=device)

            # Diffusion ε-MSE (joint or after warmup)
            if freeze_unet:
                l_eps = torch.tensor(0.0, device=device)
            else:
                t=torch.randint(0, cfg["num_timesteps"], (ct.size(0),), device=device).long()
                z_t, noise = q_sample_latent(z_mri, t, sched)
                eps_pred=net(torch.cat([z_t, z_ct],1).float(), t)
                l_eps=mse(eps_pred, noise)

            loss = l_eps + cfg["align_weight"]*l_align + cfg["contrast"]["weight"]*l_con

            opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); ema.update(net)

            if (step+1)%50==0:
                print(f"[{epoch+1}/{cfg['epochs']}] step {step+1}/{len(dl)} "
                      f"loss {loss.item():.4f} | eps {float(l_eps):.4f} | align {l_align.item():.4f} | con {float(l_con):.4f}")

        # save
        os.makedirs(cfg["save_dir"], exist_ok=True)
        torch.save({
            "model": net.state_dict(),
            "ct_encoder": E_ct.state_dict(),
            "proj_ct": Proj_ct.state_dict(),
            "proj_mri": Proj_mri.state_dict(),
            "ema": ema.shadow, "opt": opt.state_dict(), "epoch": epoch+1, "cfg": cfg
        }, os.path.join(cfg["save_dir"], "last.pt"))

        # periodic inference
        every=cfg["inference"].get("every_n_epochs",20)
        if cfg["inference"].get("enabled",False) and ((epoch+1)%every==0):
            ema.apply_shadow(net); net.eval(); E_ct.eval()
            run_epoch_inference_align(net,E_ct,vae,sched,cfg,ds,device,epoch+1)
            net.train(); E_ct.train(); ema.restore(net)

        print(f"Epoch {epoch+1} done in {time.time()-t0:.1f}s")
    print("Training finished.")

In [12]:
main(CFG)

[Dataset] discovered pairs: 9533


/tmp/ipykernel_2738594/1409531000.py:16: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  sd_all=torch.load(ckpt_path, map_location="cpu"); state=sd_all.get("state_dict", sd_al

making attention of type 'vanilla' with 512 in_channels
Working with z of shape (1, 3, 64, 64) = 12288 dimensions.
making attention of type 'vanilla' with 512 in_channels


/home/youkyoung/.conda/envs/med/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/youkyoung/.conda/envs/med/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
/home/youkyoung/CT2MRI/diffusion_model/taming/modules/losses/lpips.py:29: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github

loaded pretrained LPIPS loss from taming/modules/autoencoder/lpips/vgg.pth


/tmp/ipykernel_2738594/3627789050.py:25: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  sd_all=torch.load(ldm_ckpt_path, map_location="cpu"); state=sd_all.get("state_dict", s

[U-Net preload] loaded 0 tensors from LDM; skipped(mismatch) 278; extra in ckpt 0


/tmp/ipykernel_2738594/407438194.py:35: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ck=torch.load(cfg["resume"], map_location=device)


[Resume] ./ckpt_ct2mri_b0_ldm_align_bucket_preinit/last.pt @ epoch 50
Training finished.


In [13]:
# ==== Inference-only (CT-only, aligned LDM) ==================================
import os, glob, torch
from PIL import Image
import numpy as np

# ---------------- USER SETTINGS ----------------
CKPT_PATH  = os.path.join(CFG["save_dir"], "last.pt")  # 학습 체크포인트
CT_GLOB    = "/data/ct-mri/dataset/processed_final_5/eval/*/ct_axial/*.jpg"
OUT_DIR    = os.path.join(CFG["save_dir"], "inference_eval")
SAMPLER    = "ddim"          # "ddim" or "ddpm"
DDIM_STEPS = 200
ETA        = 0.0
BATCH      = 4
USE_CT_INIT= True            # z = z_ct + σ·ε 초기화(정렬을 활용)
DEVICE     = CFG["device"]
# ------------------------------------------------

# ---------- small helpers ----------
def imread_gray(path, size):
    from PIL import Image
    return np.asarray(Image.open(path).convert("L").resize((size,size), Image.BICUBIC), np.float32)/255.0

def to_tensor01(arr):  # (H,W)->(1,H,W)
    return torch.from_numpy(arr)[None, ...]

def to_m11(x01):       # [0,1]->[-1,1]
    return x01*2.0 - 1.0

def to_img_from_m11(x): # (1,H,W)[-1,1]->PIL
    x = x.detach().clamp(-1,1)
    x = (x+1.0)*0.5
    return Image.fromarray((x[0].cpu().numpy()*255).round().astype(np.uint8), "L")

def save_triptych(ct_m11, pred_m11, gt_path, out_path, size):
    ct_p   = to_img_from_m11(ct_m11)
    pred_p = to_img_from_m11(pred_m11)
    if gt_path and os.path.exists(gt_path):
        gt = Image.fromarray((imread_gray(gt_path, size)*255).astype(np.uint8), "L")
    else:
        gt = Image.new("L", (size,size), 0)
    panel = Image.new("L", (size*3, size))
    panel.paste(ct_p,(0,0)); panel.paste(gt,(size,0)); panel.paste(pred_p,(size*2,0))
    os.makedirs(os.path.dirname(out_path), exist_ok=True); panel.save(out_path)

# ---------- sched & samplers ----------
try:
    from ldm.modules.diffusionmodules.util import make_beta_schedule, make_ddim_timesteps, make_ddim_sampling_parameters
except Exception:
    from ldm.modules.diffusionmodules.utils import make_beta_schedule, make_ddim_timesteps, make_ddim_sampling_parameters

class Sched:
    def __init__(self, betas, alphas, a_bar, a_bar_prev):
        self.betas=betas; self.alphas=alphas
        self.a_bar=a_bar; self.a_bar_prev=a_bar_prev
        self.sqrt_a_bar=torch.sqrt(a_bar)
        self.sqrt_one_minus_a_bar=torch.sqrt(1.0 - a_bar)
        self.post_var=betas*(1.0-a_bar_prev)/(1.0-a_bar)

def build_schedule(num_steps, beta_schedule, device):
    betas  = torch.from_numpy(make_beta_schedule(beta_schedule, num_steps)).float().to(device)
    alphas = 1.0 - betas
    a_bar  = torch.cumprod(alphas, dim=0)
    a_prev = torch.cat([torch.tensor([1.0], device=device), a_bar[:-1]], dim=0)
    return Sched(betas, alphas, a_bar, a_prev)

@torch.no_grad()
def sample_ddpm_latent_aligned(net, E_ct, vae, ct_m11, sched, device="cuda", use_ct_init=True):
    z_ct = E_ct(ct_m11)
    B, Cz, h, w = z_ct.shape
    z = z_ct + torch.randn(B,Cz,h,w,device=device) if use_ct_init else torch.randn(B,Cz,h,w,device=device)
    T = sched.betas.size(0)
    for t in reversed(range(T)):
        t_b  = torch.full((B,), t, device=device, dtype=torch.long)
        eps  = net(torch.cat([z, z_ct],1).float(), t_b)
        a_t  = sched.a_bar[t]
        z0   = (z - torch.sqrt(1.0 - a_t)*eps) / torch.sqrt(a_t)
        mean = torch.sqrt(sched.a_bar_prev[t])*z0 + torch.sqrt(1.0 - sched.a_bar_prev[t])*eps
        z    = mean + (torch.sqrt(sched.post_var[t])*torch.randn_like(z) if t>0 else 0.0)
    return vae.decode(z)

@torch.no_grad()
def sample_ddim_latent_aligned(net, E_ct, vae, ct_m11, sched, ddim_steps=200, eta=0.0, device="cuda", use_ct_init=True):
    z_ct = E_ct(ct_m11)
    B, Cz, h, w = z_ct.shape
    z = z_ct + torch.randn(B,Cz,h,w,device=device) if use_ct_init else torch.randn(B,Cz,h,w,device=device)

    ts = np.array(make_ddim_timesteps('uniform', ddim_steps, sched.betas.size(0))) - 1
    sigmas, alphas, alphas_prev = make_ddim_sampling_parameters(sched.a_bar.cpu().numpy(), ts, eta)
    sigmas = torch.from_numpy(sigmas).float().to(device)
    alphas = torch.from_numpy(alphas).float().to(device)
    a_prev = torch.from_numpy(alphas_prev).float().to(device)

    for i in reversed(range(len(ts))):
        t   = int(ts[i])
        t_b = torch.full((B,), t, device=device, dtype=torch.long)
        eps = net(torch.cat([z,z_ct],1).float(), t_b)
        a_t, ap, s_t = alphas[i], a_prev[i], sigmas[i]
        z0 = (z - torch.sqrt(1.0 - a_t)*eps) / torch.sqrt(a_t)
        dir_term = torch.sqrt(1.0 - ap - s_t**2) * eps
        noise = s_t*torch.randn_like(z) if i>0 else 0.0
        z = torch.sqrt(ap)*z0 + dir_term + noise
    return vae.decode(z)

# ---------- load ckpt & rebuild ----------
assert os.path.exists(CKPT_PATH), f"ckpt not found: {CKPT_PATH}"
ckpt = torch.load(CKPT_PATH, map_location="cpu")
cfg  = ckpt["cfg"]
img_size = cfg["img_size"]

# VAE (teacher) 재구성
vae = build_vae(CFG).to(DEVICE).eval()  # build_vae(CFG)는 위 학습 때 썼던 걸 그대로 사용

# latent size 추정
ct_paths_all = sorted(glob.glob(CT_GLOB))
assert len(ct_paths_all)>0, f"No CT matched: {CT_GLOB}"
x01 = to_tensor01(imread_gray(ct_paths_all[0], img_size)).unsqueeze(0).to(DEVICE)
with torch.no_grad(): z0 = vae.encode(to_m11(x01))
lat_hw   = z0.shape[-2]
z_ch     = z0.shape[1]

# U-Net/ E_ct 복원
net  = build_latent_unet(CFG, lat_hw).to(DEVICE).eval()
E_ct = CTLatentEncoder(z_channels=z_ch).to(DEVICE).eval()

net.load_state_dict(ckpt["model"], strict=True)
E_ct.load_state_dict(ckpt["ct_encoder"], strict=True)

# EMA 적용(있으면)
if ckpt.get("ema", None) is not None:
    with torch.no_grad():
        for n,p in net.named_parameters():
            if p.requires_grad and n in ckpt["ema"]:
                p.data.copy_(ckpt["ema"][n].to(p.device))

# 스케줄
sched = build_schedule(cfg["num_timesteps"], cfg["beta_schedule"], DEVICE)

# ---------- run ----------
os.makedirs(OUT_DIR, exist_ok=True)
print(f"[INFO] matched CT files: {len(ct_paths_all)} | saving to: {OUT_DIR}")

for i in range(0, len(ct_paths_all), BATCH):
    batch_paths = ct_paths_all[i:i+BATCH]
    imgs_m11, stems, pids = [], [], []
    for p in batch_paths:
        arr = imread_gray(p, img_size)
        imgs_m11.append(to_m11(to_tensor01(arr)))
        stems.append(os.path.splitext(os.path.basename(p))[0])
        pids.append(os.path.basename(os.path.dirname(os.path.dirname(p))))  # eval/<PID>/ct_axial/<stem>.jpg
    ct_m11 = torch.stack(imgs_m11, 0).to(DEVICE)

    if SAMPLER.lower()=="ddpm":
        pred_m11 = sample_ddpm_latent_aligned(net, E_ct, vae, ct_m11, sched, device=DEVICE, use_ct_init=USE_CT_INIT)
        tag = "ddpm"
    else:
        pred_m11 = sample_ddim_latent_aligned(net, E_ct, vae, ct_m11, sched, ddim_steps=DDIM_STEPS, eta=ETA, device=DEVICE, use_ct_init=USE_CT_INIT)
        tag = f"ddim_eta{ETA:.1f}"

    for k, stem in enumerate(stems):
        base = f"{pids[k]}_{stem}_{tag}"
        to_img_from_m11(pred_m11[k]).save(os.path.join(OUT_DIR, f"{base}.png"))

        # GT: 형제 폴더에서 탐색
        pid_dir = os.path.dirname(os.path.dirname(batch_paths[k]))     # .../<PID>
        gt_path = os.path.join(pid_dir, "mri_b0_axial", f"{stem}.jpg")
        trip    = os.path.join(OUT_DIR, f"{base}_triptych.png")
        save_triptych(ct_m11[k], pred_m11[k], gt_path if os.path.exists(gt_path) else None, trip, size=img_size)

print("[Done]")


/tmp/ipykernel_2738594/2703792614.py:106: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(CKPT_PATH, map_location="cpu")
/tmp/ipykernel_2738594/1409531000.py

making attention of type 'vanilla' with 512 in_channels
Working with z of shape (1, 3, 64, 64) = 12288 dimensions.
making attention of type 'vanilla' with 512 in_channels
loaded pretrained LPIPS loss from taming/modules/autoencoder/lpips/vgg.pth
[INFO] matched CT files: 93 | saving to: ./ckpt_ct2mri_b0_ldm_align_bucket_preinit/inference_eval
Selected timesteps for ddim sampler: [  1   6  11  16  21  26  31  36  41  46  51  56  61  66  71  76  81  86
  91  96 101 106 111 116 121 126 131 136 141 146 151 156 161 166 171 176
 181 186 191 196 201 206 211 216 221 226 231 236 241 246 251 256 261 266
 271 276 281 286 291 296 301 306 311 316 321 326 331 336 341 346 351 356
 361 366 371 376 381 386 391 396 401 406 411 416 421 426 431 436 441 446
 451 456 461 466 471 476 481 486 491 496 501 506 511 516 521 526 531 536
 541 546 551 556 561 566 571 576 581 586 591 596 601 606 611 616 621 626
 631 636 641 646 651 656 661 666 671 676 681 686 691 696 701 706 711 716
 721 726 731 736 741 746 751 756 76